In [1]:
import polars as pl
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from statsmodels.regression.rolling import RollingOLS
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

from itables import show

# Load data

In [2]:
df = pl.read_parquet("1d_candle_panel_cleaned_universe.parquet").sort(["symbol", "timestamp"])

In [3]:
df = df.drop(["open", "high", "low", "trades"])

# Create price index that is rebalanced daily, weighted by sqrt(lagged rolling 30 day median dollar volume)

In [4]:
# Compute dollar volume
df = df.with_columns(
    (pl.col("volume") * pl.col("close")).over("symbol").alias("dollar_volume")
).sort(["symbol", "timestamp"])

# Compute 30 day rolling median dollar volumeabs
df = df.with_columns(
    pl.col("dollar_volume").rolling_median(window_size=30, min_samples=30).shift(1).over("symbol").alias("lagged_30_mdv")
).drop_nans().drop_nulls().sort(["symbol", "timestamp"])

# Compute index arithmetric returns

In [5]:
index = df.group_by(
    "timestamp", maintain_order=True
).agg(
    ((pl.col("log_return").exp() - 1) * pl.col("lagged_30_mdv").sqrt() / pl.col("lagged_30_mdv").sqrt().sum())
    .sum().alias("index_simple_return")
).sort("timestamp")

In [6]:
index = index.with_columns(
    (pl.col("index_simple_return") + 1).log().alias("index_log_return")
).sort("timestamp").with_columns(
    pl.col("index_log_return").cum_sum().exp().alias("index_close")
).sort("timestamp")

In [7]:
index.plot.line(
    y="index_close",
    x="timestamp"
)

alt.Chart(...)

# Perform rolling OLS to estimate betas of symbols to the index

In [8]:
# Join Index log_return to DF 
test = df.join(
    other=index.select("timestamp", "index_log_return"),
    on="timestamp",
    how="left"
).sort(["symbol", "timestamp"])

In [9]:
def beta_regression(df, symbol):
    reg_df = df.filter(pl.col("symbol") == symbol).sort("timestamp")
    x = reg_df.select("index_log_return")
    X = add_constant(x)
    y = reg_df.select("log_return")

    model = RollingOLS(y, X, window=240, min_nobs=240)
    out = pl.from_pandas(model.fit().params["x1"]).rename(symbol)
    return out

In [12]:
beta_regression(test, "kPEPE")

kPEPE
f64
null
null
null
null
null
…
1.290452
1.28974
1.291198
